<div style="line-height:18px;">
    &emsp;&emsp;&emsp;
    <br>
    &emsp; <i><u>Prof. Dr. Caetano Traina Júnior</u><br>
    &emsp; &ensp;<i>ICMC-USP São Carlos
<div align="right">3 seg. &nbsp; 28 cel</div>
    </div><br>

<b>7 - Executar consultas sobre dados de tipo <b>HStore,</b><br>
<b> &emsp; &nbsp; estendendo a base de dados de <u>Alunos80K</u></b>
        
<br><br>

<br>

**Objetivos:** 
  * Usar o SGBD  para aprender a usar o tipo de dados 'HSTORE'
  * Acrescentar uma nova tabela: `Configura` (sintética),\
    com os parâmetros de configurações individuais de um aplicativo,\
    na Base de Dados: `Alunos80K`
    * Essa tabela tem um atributo do tipo `HSTORE`
  * Aprender a usar operações de atualização em um atributo de tipo `HSTORE`
  * Executar consultas sobre o tipo de dados `HSTORE`.

<br><br>

# 1. Conectar com a Base de Dados

Para começar a explorar uma base, é necessário estabelecer antes a conexão com o SGBD onde ela está armazenada:<br>


In [ ]:
# Importamos os módulos que usamos no notebook.
from ipywidgets import interact
import ipywidgets as widgets
from sqlalchemy import create_engine


# Conectamos ao banco alunos80k com nosso usuário e senha.
%load_ext sql

# Informamos usuário, senha, servidor, porta e banco na conexão.
engine = create_engine('postgresql://postgres:minha senha@localhost/alunos80k')
%sql postgresql://postgres:minha senha@localhost:5432/alunos80k
%config SqlMagic.displaylimit=None




<br>

# 6.2. Módulos de extensão em Postgres

O Gerenciador  dispõe de diversos módulos que podem estender sua funcionalidade.\
 * Alguns já são trazidos pelo processo de instalação usual do gerenciador.\
   Nesses casos, eles podem ser simplesmente ativados.\
   <small>(uma vez ativados, permanecem assim até serem explicitamente desativados)</small>
 * Outros precisam ser buscados (em geral, em seu código-fonte).\
   Nesses casos, eles precisam ser compilados junto com o servidor, \
   para serem depois ativados.

Ativar e desativar extensões usam os comandos da `DDL`:

<div class="alert alert-block alert-warning">
    Sintaxe para <b>ativar uma extensão:</b>
    <br>
    
        CREATE EXTENSION [IF NOT EXISTS]  $<$nome$>$;
        <br><br>
    Sintaxe para <b>desativar uma extensão:</b>
    <br>
    
        DROP EXTENSION [IF EXISTS]  $<$nome$>$;
        
</div>

Os módulos de extensão adicionais 
  * fornecidos com a instalação-padrão do \
   &emsp; &emsp;   estão documentados no <a href="https://www.postgresql.org/docs/current/contrib.html"> Manual do Postgres</a>.
  * mantidos pela rede de desenvolvedores de software aberto `PostgreSQL Extension Network (PGXN)`\
   &emsp; &emsp;   estão documentados no <a href="https://www.postgresql.org/docs/current/contrib.html"> site da PGXN</a>.

<br>

---

<br>

Dados do tipo `HSTORE` podem ser acessados em  pela <b>Extensão `HStore`.</b>\
Como vamos usar dados desse tipo, é necessário ativá-la:

In [2]:
%%sql
CREATE EXTENSION IF NOT EXISTS hstore;

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
Done.


[]

<br><br> 

## 2.1. Verificar as extensões que estão ativas

O catálogo do gerenciador  contém 
a tabela `PG_Extension`, que mantém dados sobre as extensões ativas.

Podemos consultar essa tabela para verificar as extensões que estão disponíveis:

In [3]:
%%sql
select * from pg_extension;

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
2 rows affected.


oid,extname,extowner,extnamespace,extrelocatable,extversion,extconfig,extcondition
14236,plpgsql,10,11,False,1.0,None,None
17937,hstore,16384,2200,True,1.8,None,None


<br>

----

<br><br>

# 3. Acrescentar um atributo de tipo `HSTORE` na tabela `Alunos`

Antes de qualquer coisa, vamos verificar os atributos que constituem a tabela `Aluno`:

In [4]:
%%sql
SELECT A.AttName, A.AttNum, T.TypName
    FROM Pg_Class C JOIN Pg_Attribute A ON C.OID = A.AttRelId
                    JOIN Pg_Type T      ON A.AttTypId=T.OID
    WHERE (C.RelName = 'alunos')
      AND A.AttNum>0   -- Consideramos apenas os atributos definidos na tabela.
    ORDER BY 2;

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
4 rows affected.


attname,attnum,typname
nusp,1,numeric
nome,2,varchar
idade,3,int4
cidade,4,varchar


<br>

A seguir, vamos acrescentar um atributo para ser usado para armazenar dados gerais para cada aluno:<br>
 <big> &emsp; Dados não previstos! <br>
  &emsp; que podem ser completamente diferentes para cada aluno!</big>

Vamos criar um atributo chamado `Dados` de tipo `HSTORE`:

In [5]:
%%sql
ALTER TABLE Alunos
    DROP COLUMN IF EXISTS Dados;
ALTER TABLE Alunos
    ADD COLUMN Dados HSTORE;

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
Done.
Done.


[]

<br>

E vamos verificar como ficou a definição da tabela:

In [6]:
%%sql
SELECT A.AttName, A.AttNum, T.TypName
    FROM Pg_Class C JOIN Pg_Attribute A ON C.OID = A.AttRelId
                    JOIN Pg_Type T      ON A.AttTypId=T.OID
    WHERE (C.RelName = 'alunos')
      AND A.AttNum>0   -- Consideramos apenas os atributos definidos na tabela.
    ORDER BY 2;

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
5 rows affected.


attname,attnum,typname
nusp,1,numeric
nome,2,varchar
idade,3,int4
cidade,4,varchar
dados,5,hstore


<br>

Como a tabela `Alunos` é aleatória, não sabemos quais alunos existem.<br>
Vamos verificar quais alunos se chamam `José da Silva`:

In [7]:
%%sql
SELECT * 
    FROM Alunos
    WHERE Nome ~* 'José.*da Silva'

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
10 rows affected.


nusp,nome,idade,cidade,dados
98210895,José da Silva,27,São Paulo-SP,None
25321757,José da Silva,21,Brasília-DF,None
56364340,José da Silva,27,Piracicaba-SP,None
93718364,José da Silva,23,Taboão da Serra-SP,None
69523110,José da Silva,20,São Paulo-SP,None
50706711,José da Silva,32,Ribeirão Preto-SP,None
96515457,José da Silva,21,Itapetininga-SP,None
48543088,José da Silva,27,São Paulo-SP,None
63260089,José da Silva,22,Divinópolis-MG,None
87042992,José da Silva,26,Regente Feijó-SP,None


<br>

Vamos inserir um novo aluno chamado `José Mário da Silva`,<br>
informando que ele nasceu em `Bueno de Andrade` e veio transferido do `ITA`.<br>

Como transferência e cidade natal não são atributos da tabela `Alunos`, essa informação é colocada no atributo `Dados`.<br>
Aproveitamos o comando de inserção e retornamos a tupla inserida inteira (com a cláusula `RETURNING *`).<br>
&emsp;<small>(primeiro apagando o aluno caso ele já exista!)</small>

In [8]:
%%sql
DELETE FROM Alunos
    WHERE NUSP=1234321;
INSERT INTO Alunos(NUSP, Nome, Idade, Cidade, Dados)
    VALUES (1234321, 'José Mário Gomes da Silva', 25, 'Araraquara-SP',
            '"Cidade Natal" => "Bueno de Andrade",
             "Transferido de"=> "ITA"')
    RETURNING *;

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
0 rows affected.
1 rows affected.


nusp,nome,idade,cidade,dados
1234321,José Mário Gomes da Silva,25,Araraquara-SP,"""Cidade Natal""=>""Bueno de Andrade"", ""Transferido de""=>""ITA"""


<br>

Vamos ver quem tem nome começando com `J` e é de `Araraquara`:

In [9]:
%sql Query <<                  \
SELECT * FROM Alunos            \
    WHERE Cidade ~* 'Araraquara' \
      AND Nome ~'^J'              \
      ORDER BY NUSP;

print(Query)
N1=Query[0][0]
N2=Query[1][0]

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
16 rows affected.
Returning data to local variable Query
+----------+---------------------------+-------+---------------+-------------------------------------------------------------+
|   nusp   |            nome           | idade |     cidade    |                            dados                            |
+----------+---------------------------+-------+---------------+-------------------------------------------------------------+
| 1234321  | José Mário Gomes da Silva |   25  | Araraquara-SP | "Cidade Natal"=>"Bueno de Andrade", "Transferido de"=>"ITA" |
| 13266003 |       Jasmim Milhães      |   39  | Araraquara-SP |                             None                            |
| 19455586 |         Jani Souto        |   21  | Araraquara-SP |                             None                            |
| 23538220 |        Jacó Gardner       |   24  | Araraquara-SP |                             None                      

<br>

De onde vieram os alunos que têm `Bueno de Andrade` como `Cidade Natal` ?

In [10]:
%%sql
SELECT Nome, Dados->'Transferido de' "veio de"
    FROM Alunos
    WHERE Dados->'Cidade Natal'='Bueno de Andrade';

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
1 rows affected.


nome,veio de
José Mário Gomes da Silva,ITA


<br>

## 3.1. Operações de atualização de valores de tipo `HStore`: `UPDATE`

Podemos acrescentar um novo par `<chave, valor>` em um atributo 'HSTORE'<br>
usando o operador de concatenação  `||` num comando `UPDATE`:

In [11]:
%%sql
UPDATE Alunos SET Dados = COALESCE(Dados,'') ||'"Trancamento" => "2''Sem. 2020"' WHERE NUSP=:N1 OR NUSP=:N2
    RETURNING *;

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
2 rows affected.


nusp,nome,idade,cidade,dados
13266003,Jasmim Milhães,39,Araraquara-SP,"""Trancamento""=>""2'Sem. 2020"""
1234321,José Mário Gomes da Silva,25,Araraquara-SP,"""Trancamento""=>""2'Sem. 2020"", ""Cidade Natal""=>""Bueno de Andrade"", ""Transferido de""=>""ITA"""


<br>
<b>Porque usar `COALESCE`?</b><br><br>

Lembre-se de que qualquer operação com `NULO` resulta em `NULO`.<br>
Portanto, a expressão `Dados ||'"Trancamento" => "2'Sem. 2020"` resulta `NULO` quando `Dados` estiver `NULO`.

Usamos a função `COALESCE(A,B)` para resolver esse problema.<br>
Ela retorna `A` quando `A` não é `Nulo`, e retorna `B`quando `A` é `Nulo`.<br>
Portanto, `COALESCE(Dados,'') ||'"Trancamento" => "2''Sem. 2020"'` acrescenta essa chave ao atributo `Dados`, seja ele `NULO` ou não.

<br>

Se não for usado o `COALESCE`, a operação de atualização ocorre apenas nas tuplas em que o atributo `Dados` está `NOT NULL`:

In [12]:
%%sql
UPDATE Alunos SET Dados = Dados ||'"Viagem de excursão" => "Fábrica da Lupo"' 
    WHERE Cidade ~* 'Araraquara'  AND Nome ~'^J' 
    RETURNING *;

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
16 rows affected.


nusp,nome,idade,cidade,dados
94057866,Jitendra Correia,23,Araraquara-SP,None
86150607,Jitendra Maiocchi,22,Araraquara-SP,None
99011739,Juliano Coelho,21,Araraquara-SP,None
25027903,Jácome Tosetto,24,Araraquara-SP,None
23538220,Jacó Gardner,24,Araraquara-SP,None
19455586,Jani Souto,21,Araraquara-SP,None
37225775,Juliano Mitidieri,28,Araraquara-SP,None
74673559,Jax César,23,Araraquara-SP,None
49314280,Jamie Vilalobos,25,Araraquara-SP,None
44988182,Jair Álvaro,28,Araraquara-SP,None


<br>

Por outro lado, como cada chave pode ter um valor só, ele pode ser trocado simplesmente com:

In [13]:
%%sql
UPDATE Alunos SET Dados = COALESCE(Dados,'') ||'"Trancamento" => "1''Sem. 2025"' WHERE NUSP=:N1;

SELECT * FROM Alunos
    WHERE Cidade ~* 'Araraquara'
      AND Nome ~'^J'
      ORDER BY NUSP;

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
1 rows affected.
16 rows affected.


nusp,nome,idade,cidade,dados
1234321,José Mário Gomes da Silva,25,Araraquara-SP,"""Trancamento""=>""1'Sem. 2025"", ""Cidade Natal""=>""Bueno de Andrade"", ""Transferido de""=>""ITA"", ""Viagem de excursão""=>""Fábrica da Lupo"""
13266003,Jasmim Milhães,39,Araraquara-SP,"""Trancamento""=>""2'Sem. 2020"", ""Viagem de excursão""=>""Fábrica da Lupo"""
19455586,Jani Souto,21,Araraquara-SP,None
23538220,Jacó Gardner,24,Araraquara-SP,None
25027903,Jácome Tosetto,24,Araraquara-SP,None
37225775,Juliano Mitidieri,28,Araraquara-SP,None
39689365,Jared Paz,21,Araraquara-SP,None
44988182,Jair Álvaro,28,Araraquara-SP,None
49314280,Jamie Vilalobos,25,Araraquara-SP,None
62413090,Joseph Carpenter,21,Araraquara-SP,None


<br>

## 3.2. Operações de remoção de pares chave-valor em atributos de tipo `HStore`

Remover uma chave de um atributo que tenha diversas chaves pode ser solicitado usando<br>
&emsp; a função `Delete`.

Por exemplo:

In [14]:
%%sql
UPDATE Alunos 
    SET Dados = Delete(Dados, 'Viagem de excursão')
    WHERE NUSP =12915664
    RETURNING *;

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
0 rows affected.


nusp,nome,idade,cidade,dados


<br>

Esse comando não atualizou nenhum aluno, pois nenhum tem o `NUSP` indicado.

<br><br>

----

<br><br>

<br><br>

# 4. Trabalhando com atributos `HSTORE`
 
## 4.1. Criar uma tabela com um atributo  `HSTORE`

<table  align="left">
  <tr>
      <td align="left">
          Vamos criar uma nova tabela chamada `Configura` que armazena, para cada `Aluno`, &emsp; <br>
          os parâmetros de configuração de um aplicativo hipotético.
      </td>
      <td rowspan="2" ></td>
  </tr>
  <tr>
      <td>
A tabela deverá ter:<br>
  &bullet; o `NUSP` do aluno como chave,<br>
  &bullet;  o `Nome` do aluno como atributo 'tradicional' e<br>
  &bullet;  um atributo de tipo `HStore` com os parâmetros de configuração de um <br>
      &emsp; aplicativo hipotético que o aluno poderia estar usando...<br>
      &emsp; &emsp; &emsp; Por exemplo: um editor de texto.
      </td>
  </tr>
  <tr>
      <td colspan="2">Note que um atributo `HStore` armazena pares de $<$ `Chave, Valor` $>$, onde cada `Chave` pode ser criada livremente.
      </td>
  </tr>
</table>
<br>


In [15]:
%%sql
DROP TABLE IF EXISTS Configura;
CREATE TABLE Configura (
    NUSP NUMERIC(8),
    Nome TEXT,
    Config HSTORE
);

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
Done.
Done.


[]

<br>

## 4.2. Carregar a tabela com os atributos _tradicionais_

A tabela `Configura` corresponde aos alunos armazenados na tabela `Aluno`.
Portanto:
  * a tabela `Configura` pode conter no atributo `NUSP` apenas `NUSP` que existam na tabela de `Alunos`,<br>
  * mas nem todo aluno precisa ter um registro de configuração.

<big>Vamos simular isso <u>carregando aleatoriamente 50% dos alunos da tabela `Alunos` na tabela  `Configura`</u>:</big>


In [16]:
%%sql
INSERT INTO  Configura(NUSP, Nome)
    SELECT NUSP, Nome
        FROM Alunos
        ORDER BY Random()
        LIMIT 40000;

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
40000 rows affected.


[]

<br>

Veja que apenas os atributos `NUSP` e `Nome` foram inicializados.\
O atributo `Config` está nulo em todas as tuplas.

In [17]:
%%sql
SELECT * FROM Configura ORDER BY Random() LIMIT 4;

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
4 rows affected.


nusp,nome,config
51596674,Salomão Pino,None
84629707,Polly Veloza,None
88680557,Ferdinando Pêcego,None
40900653,Melânia Woods,None


## 4.3. Atualizar a tabela com pares chave-valor (aleatoriamente)

O atributo `Config` é do tipo `HSTORE`.<br>
Como este é um exemplo de manipulação de valores numa Base de Dados, e numa tabela hipotética,<br>
vamos carregar esse atributo com valores aleatórios, assumindo as seguintes `chaves`:
  * <big>`Line` e `Col`:</big> correspondem à linha e coluna de um arquivo que o aluno estiver editando\
    &mdash; assumimos que 80% dos alunos têm essas chaves.\
    &emsp; Esses dois atributos devem sempre ocorrer juntos.
  * <big>`BkGrdnColor`, `FrGrdnColor`:</big> cor do texto e do fundo\
    &mdash; assumimos que 60% dos alunos têm essas chaves.\
    &emsp; Esses dois atributos devem sempre ocorrer juntos.
  * <big>`Font`:</big> fonte da letra sendo usada\
  *  &mdash; assumimos que 60% dos alunos têm essas chaves.
  * <big>`Proxi`:</big> se está conectado via um _proxy_
     &mdash; assumimos que 50% dos alunos têm essa chave como 'false'<br>
     &emsp;  e 50% dos restantes têm essa chave como 'true'.

Essa especificação pode ser carregada com o seguinte <i>script</i>:

In [18]:
%%sql
-- Geramos os valores de line e col juntos para manter o par.
UPDATE Configura 
    SET Config = HStore(ARRAY['line', 'col'], 
                       ARRAY[TEXT((300*Random()+20*Random())::INT), TEXT((40*Random()+20*Random()+10*Random())::INT)])
    WHERE Random()<.8;

-- Geramos as cores de fundo e de texto juntas.
UPDATE Configura 
    SET Config = HStore(ARRAY['BkGrdnColor', 'FrGrdnColor'], 
                                 ARRAY[TEXT((16777215*Random())::INT), TEXT((16777215*Random())::INT)])
    WHERE Config IS NULL AND Random()<.6;
UPDATE Configura 
    SET Config = Config || HStore(ARRAY['BkGrdnColor', 'FrGrdnColor'], 
                                 ARRAY[TEXT((16777215*Random())::INT), TEXT((16777215*Random())::INT)])
    WHERE Config IS NOT NULL AND Random()<.6;

-- Escolhemos uma fonte para parte dos alunos.
CREATE TEMPORARY TABLE Fonts (I INT, F TEXT);
INSERT INTO Fonts VALUES(1, 'Times'), (2, 'Helvetica'), (3, 'Arial'), (4, 'Courier'), (5, 'Garamond');

UPDATE Configura
    SET Config = COALESCE(Config, ''::hstore) || hstore('Font', (SELECT F FROM Fonts WHERE I=(NUSP%5+1))::TEXT)
    WHERE Random()<.6;

DROP TABLE Fonts;

-- Geramos os valores true e false para a chave Proxy.
UPDATE Configura SET Config=Config || HStore('Proxy', 'true')
    WHERE Random()<.5;
UPDATE Configura SET Config=Config || HStore('Proxy', 'false')
    WHERE Random()<.5;

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
31994 rows affected.
4791 rows affected.
22054 rows affected.
Done.
5 rows affected.
24009 rows affected.
Done.
20016 rows affected.
20021 rows affected.


[]

<br>

Veja que:<ol>
  <li> Atributos que devem existir em pares (sincronizados) devem ser atualizados em pares (`line` e `col`, `FrGrdnColor` e `FrGrdnColor`). <br><br>
  <li> A construção `SET atrib=valor` atribui o novo valor, portanto, destrói qualquer chave que poderia existir e o substitui pelas novas.<br><br>
  <li> A construção `SET atrib=atrib || valor` concatena o novo valor aos já existentes.<br>
    Mas se `atrib` estava nulo, `atrib || nulo` resulta nulo, e não atribui nada!<br>
    Vide o caso das chaves `FrGrdnColor` e `FrGrdnColor`.<br><br>
  <li> É possível atribuir um valor diretamente a uma chave, independentemente do atributo estar ou não nulo.<br>
    Vide o caso da chave `Font`.<br><br>
  <li> Se o atributo já tivesse uma chave e um novo valor é colocado para a mesma chave, o valor anterior é substituído pelo novo (cada chave só pode ter um valor)<br>
    Vide o caso da chave `Proxy`.
  </ol>

Podemos ver rapidamente algumas das tuplas geradas:

In [19]:
%%sql
SELECT * FROM Configura
    ORDER BY RANDOM()
    LIMIT 20;

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
20 rows affected.


nusp,nome,config
55527463,Alcides Festas,"""col""=>""58"", ""line""=>""182"", ""Proxy""=>""true"""
21697674,Avril Casado,"""col""=>""15"", ""line""=>""17"", ""Proxy""=>""true"", ""BkGrdnColor""=>""13838257"", ""FrGrdnColor""=>""1987747"""
58819732,Melodia Stephens,None
90716998,Tolentino Rangel,"""col""=>""48"", ""line""=>""189"", ""Proxy""=>""false"""
32626556,Idelso Balistiero,"""col""=>""32"", ""line""=>""128"", ""Proxy""=>""false"", ""BkGrdnColor""=>""3323252"", ""FrGrdnColor""=>""1551996"""
73806317,Sandro Madeu,"""col""=>""35"", ""Font""=>""Arial"", ""line""=>""107"", ""Proxy""=>""true"""
40665051,Jensen Penha,"""col""=>""56"", ""Font""=>""Helvetica"", ""line""=>""292"", ""Proxy""=>""true"""
99009948,Jorge Sardinha,"""col""=>""47"", ""Font""=>""Courier"", ""line""=>""180"", ""Proxy""=>""false"""
42830608,Celisia Gil,"""col""=>""28"", ""Font""=>""Courier"", ""line""=>""51"", ""Proxy""=>""false"""
18947022,Jonathan Pavía,None


<br>

## 4.4. Executar consultas com atributos de tipo `HSTORE`

Podemos tanto:
  * Recuperar o valor das chaves de um atributo `HSTORE`:
    * o fonte e a cor do texto `Font` e `FrGrdnColor`
  * Quanto usar o valor das chaves para comparações
    * por exemplo, na cláusula `WHERE`:
    * o fonte  `Font`

Por exemplo:<br>
<i>Selecionar os alunos que trabalham com o _font_ &nbsp;'`Arial`'</i>:

In [20]:
%%sql
SELECT NUSP, Nome, 
       Config->'Font' Font,
       To_Hex((Config->'FrGrdnColor')::INT) FrGrdnColor,
       Config
    FROM Configura
    WHERE Config->'Font' = 'Arial'
    LIMIT 5

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
5 rows affected.


nusp,nome,font,frgrdncolor,config
24045427,Aliana Candeias,Arial,None,"""Font""=>""Arial"""
59821617,Olinto Beck,Arial,None,"""Font""=>""Arial"""
85116597,Rapaz Diegues,Arial,None,"""Font""=>""Arial"", ""Proxy""=>""true"""
16950392,Bertolino Bivar,Arial,None,"""Font""=>""Arial"", ""Proxy""=>""false"""
41825522,Evelásio Dal Canal,Arial,48197f,"""Font""=>""Arial"", ""Proxy""=>""false"", ""BkGrdnColor""=>""12513319"", ""FrGrdnColor""=>""4725119"""


<br>

Podemos pedir uma lista de quais são as `Chaves` de um atributo `HSTORE`, sem nos interessar pelos valores correspondentes,<br>
usando a função `AKeys` (_All Keys_):

In [21]:
%%sql
SELECT Nome, AKeys(Dados) FROM Alunos
    WHERE Cidade ~* 'Araraquara'
      AND Nome ~'^José'
      ORDER BY NUSP;

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
1 rows affected.


nome,akeys
José Mário Gomes da Silva,"['Trancamento', 'Cidade Natal', 'Transferido de', 'Viagem de excursão']"


<br>

Podemos pedir quais são as `Chaves` de um atributo `HSTORE`, mas fornecendo o resultado como um conjunto de dados<br>
&emsp; usando a função `Each`:

In [22]:
%%sql
SELECT Nome, SKeys(Dados) FROM Alunos
    WHERE Cidade ~* 'Araraquara'
      AND Nome ~'^José'
      ORDER BY NUSP;

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
4 rows affected.


nome,skeys
José Mário Gomes da Silva,Trancamento
José Mário Gomes da Silva,Cidade Natal
José Mário Gomes da Silva,Transferido de
José Mário Gomes da Silva,Viagem de excursão


<br>

E podemos obter o resultado tanto das chaves quanto dos valores como um conjunto de dados<br>
usando a função `Each`:

In [23]:
%%sql
SELECT Nome, 
       Each(Dados) "key-value",
       (Each(Dados)).*
    FROM Alunos
    WHERE Cidade ~* 'Araraquara'
      AND Nome ~'^José'
      ORDER BY NUSP;

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
4 rows affected.


nome,key-value,key,value
José Mário Gomes da Silva,"(Trancamento,""1'Sem. 2025"")",Trancamento,1'Sem. 2025
José Mário Gomes da Silva,"(""Cidade Natal"",""Bueno de Andrade"")",Cidade Natal,Bueno de Andrade
José Mário Gomes da Silva,"(""Transferido de"",ITA)",Transferido de,ITA
José Mário Gomes da Silva,"(""Viagem de excursão"",""Fábrica da Lupo"")",Viagem de excursão,Fábrica da Lupo


<br>

E claro, podemos também pedir somente os valores como um conjunto de dados<br>
 &emsp; usando as funções `AVals` e `SVals`:

In [24]:
%sql Q1 <<                        \
SELECT Nome,                      \
       AVals(Dados)               \
    FROM Alunos                   \
    WHERE Cidade ~* 'Araraquara'  \
      AND Nome ~'^José'           \
      ORDER BY NUSP;

%sql Q2 <<                        \
SELECT Nome,                      \
       SKeys(Dados),              \
       SVals(Dados)               \
    FROM Alunos                   \
    WHERE Cidade ~* 'Araraquara'  \
      AND Nome ~'^José'           \
      ORDER BY NUSP;

print(Q1)
print(Q2)

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
1 rows affected.
Returning data to local variable Q1
 * postgresql://postgres:minha senha@localhost:5432/alunos80k
4 rows affected.
Returning data to local variable Q2
+---------------------------+---------------------------------------------------------------+
|            nome           |                             avals                             |
+---------------------------+---------------------------------------------------------------+
| José Mário Gomes da Silva | ["1'Sem. 2025", 'Bueno de Andrade', 'ITA', 'Fábrica da Lupo'] |
+---------------------------+---------------------------------------------------------------+
+---------------------------+--------------------+------------------+
|            nome           |       skeys        |      svals       |
+---------------------------+--------------------+------------------+
| José Mário Gomes da Silva |    Trancamento     |   1'Sem. 2025    |
| José Mário Gomes d

<br>

Podemos converter um valor `HSTORE` em um valor 'JSON', <br>
 &emsp; usando a função `HStore_TO_Json':

In [25]:
%%sql
SELECT Nome, 
       HStore_TO_Json(Dados)
    FROM Alunos
    WHERE Cidade ~* 'Araraquara'
      AND Nome ~'^José'
      ORDER BY NUSP;

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
1 rows affected.


nome,hstore_to_json
José Mário Gomes da Silva,"{'Trancamento': ""1'Sem. 2025"", 'Cidade Natal': 'Bueno de Andrade', 'Transferido de': 'ITA', 'Viagem de excursão': 'Fábrica da Lupo'}"


<br>

Podemos perguntar pelas tuplas que têm alguma chave definida, sem nos importar com o valor.<br>
 &emsp; Isso é feito com o operador `?`.

Por exemplo, vamos verificar algumas contagens sobre os valores inseridos:

In [26]:
%%sql
SELECT Count(*) "Tuplas total",
       Count(*) FILTER (WHERE Config IS NULL) "config nula",
       Count(*) FILTER (WHERE Config ? 'line') "config line",
       Count(*) FILTER (WHERE Config ? 'col') "config col",
       Count(*) FILTER (WHERE Config ? 'FrGrdnColor') "config FrGrdnColor",
       Count(*) FILTER (WHERE Config ? 'BkGrdnColor') "config BkGrdnColor",
       Count(*) FILTER (WHERE Config ? 'Font') "config Font",
       Count(*) FILTER (WHERE Config ? 'Proxy') "config Proxy",
       Count(*) FILTER (WHERE Config -> 'Proxy' = 'true') "Proxy True",
       Count(*) FILTER (WHERE Config -> 'Proxy' = 'false') "Proxy False"
    FROM Configura;

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
1 rows affected.


Tuplas total,config nula,config line,config col,config FrGrdnColor,config BkGrdnColor,config Font,config Proxy,Proxy True,Proxy False
40000,1291,31994,31994,23964,23964,24009,29000,9595,19405


<br>

Os operadores `?&` e `?|` recuperam tuplas cujo valor do atributo `HSTORE` tem, respectivamente:
  * `?&`: todos os valores de uma lista de valores;
  * `?|`: pelo menos um valor de uma lista de valores;

Por exemplo, podemos verificar quantas tuplas têm:
  * `Line` e `col`
  * `Line` ou `col`
  * `Line` e `FrGrdnColor`
  * `Line` ou `FrGrdnColor`

In [27]:
%%sql
SELECT Count(*) FILTER (WHERE Config ?& ARRAY['line', 'col']) "Line e col",
       Count(*) FILTER (WHERE Config ?| ARRAY['line', 'col']) "Line ou col",
       Count(*) FILTER (WHERE Config ?& ARRAY['line', 'FrGrdnColor']) "Line e FrGrdnColor",
       Count(*) FILTER (WHERE Config ?| ARRAY['line', 'FrGrdnColor']) "Line ou FrGrdnColor"
    FROM Configura;

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
1 rows affected.


Line e col,Line ou col,Line e FrGrdnColor,Line ou FrGrdnColor
31994,31994,19173,36785


<br><br>

# 8. Finalização

Para finalizar o <i>notebook</i>, sempre é conveniente deixar a base de dados como estava no início.<br>
Neste <i>notebook</i>, devemos <ul>
  <li> Remover o atributo `Dados` da tabela `Alunos`
  <li> Apagar o aluno de `NUSP=1234321`.
  </ul>

In [28]:
%%sql
ALTER TABLE Alunos
    DROP COLUMN IF EXISTS Dados;

DELETE FROM Alunos
    WHERE NUSP=1234321;

 * postgresql://postgres:minha senha@localhost:5432/alunos80k
Done.
1 rows affected.


[]

<br>

A tabela `Configura` também foi criada neste <i>notebook</i>.<br>
No entanto, queremos mantê-la para usá-la no <B>Exercício da semana</B>.

Portanto, a célula seguinte está desabilitada.<br>
Ela pode ser habilitada se quisermos remover essa tabela também.

<br><br>


    <b>Estender a base de dados de <u>Alunos80K</u> e <br>executar consultas sobre dados de tipo <b>HSTORE</b>
    <br>


    &emsp;&emsp;&nbsp;
    <b>FIM</b>&nbsp;&nbsp;&nbsp;&nbsp;
    
    

<br><br>

<br>
